<a href="https://colab.research.google.com/github/joostmeyer26/AI_Engineer_Project_4/blob/main/Mercedes_Benz_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course 5 Project Mercedes Benz
# Project Objective:  
# Reduce the time a Mercedes-Benz spends on the test bench.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
from google.colab import files

# Select both train.csv and test.csv from your computer.
uploaded = files.upload()

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Training data:", train_df.shape)
print("Test data:", test_df.shape)

train_df.head()

Saving test.csv to test (1).csv
Saving train.csv to train (1).csv
Training data: (4209, 378)
Test data: (4209, 377)


,ID,y,X0,X1,X2,X3,X4,X5,X6,X8,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,0,130.81,k,v,at,a,d,u,j,o,...,0,0,1,0,0,0,0,0,0,0
1,6,88.53,k,t,av,e,d,y,l,o,...,1,0,0,0,0,0,0,0,0,0
2,7,76.26,az,w,n,c,d,x,j,x,...,0,0,0,0,0,0,1,0,0,0
3,9,80.62,az,t,n,f,d,x,l,e,...,0,0,0,0,0,0,0,0,0,0
4,13,78.02,az,v,n,f,d,h,d,n,...,0,0,0,0,0,0,0,0,0,0


In [4]:
# Summarize every column in each dataset.
train_summary = pd.DataFrame({
    "Missing values": train_df.isna().sum(),
    "Unique values": train_df.nunique()
})

test_summary = pd.DataFrame({
    "Missing values": test_df.isna().sum(),
    "Unique values": test_df.nunique()
})

print("Total missing values in training:", train_summary["Missing values"].sum())
print("Total missing values in test:", test_summary["Missing values"].sum())

print("\nTraining column summary:")
display(train_summary)

print("\nTest column summary:")
display(test_summary)

Total missing values in training: 0
Total missing values in test: 0

Training column summary:


,Missing values,Unique values
ID,0,4209
y,0,2545
X0,0,47
X1,0,27
X2,0,44
...,...,...
X380,0,2
X382,0,2
X383,0,2
X384,0,2



Test column summary:


,Missing values,Unique values
ID,0,4209
X0,0,49
X1,0,27
X2,0,45
X3,0,7
...,...,...
X380,0,2
X382,0,2
X383,0,2
X384,0,2


In [5]:
# Separate the target and exclude the identifier.
y = train_df["y"].copy()
X = train_df.drop(columns=["ID", "y"]).copy()

# Save test IDs so we can match predictions to each car.
test_ids = test_df["ID"].copy()
X_test = test_df.drop(columns=["ID"]).copy()

# Find columns with only one unique training value.
constant_columns = X.columns[X.nunique() == 1].tolist()

print("Number of constant columns:", len(constant_columns))
print("Columns to remove:", constant_columns)

# Remove the SAME columns from both datasets.
X = X.drop(columns=constant_columns)
X_test = X_test.drop(columns=constant_columns)

print("\nTraining features:", X.shape)
print("Test features:", X_test.shape)

Number of constant columns: 12
Columns to remove: ['X11', 'X93', 'X107', 'X233', 'X235', 'X268', 'X289', 'X290', 'X293', 'X297', 'X330', 'X347']

Training features: (4209, 364)
Test features: (4209, 364)


In [6]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = [
    "X0", "X1", "X2", "X3", "X4", "X5", "X6", "X8"
]

# Keep the original features and create encoded copies.
X_encoded = X.copy()
X_test_encoded = X_test.copy()

encoders = {}

for column in categorical_columns:
    encoder = LabelEncoder()

    # Include categories from BOTH datasets, as the assignment requires.
    combined_values = pd.concat(
        [X[column], X_test[column]],
        ignore_index=True
    )

    encoder.fit(combined_values)

    # Apply the same mapping to training and test values.
    X_encoded[column] = encoder.transform(X[column])
    X_test_encoded[column] = encoder.transform(X_test[column])

    encoders[column] = encoder

print("Encoding complete!")
display(X_encoded[categorical_columns].head())

Encoding complete!


,X0,X1,X2,X3,X4,X5,X6,X8
0,37,23,20,0,3,27,9,14
1,37,21,22,4,3,31,11,14
2,24,24,38,2,3,30,9,23
3,24,21,38,5,3,30,11,4
4,24,23,38,5,3,14,3,13


# Encoding succeeded. We reserve 20% of the labeled training data for validation.

# Training portion: Used to learn the model.
Validation portion: Used to measure prediction errors.
random_state=42: Makes the random split repeatable

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Validation features:", X_valid.shape)
print("Training targets:", y_train.shape)
print("Validation targets:", y_valid.shape)

Training features: (3367, 364)
Validation features: (842, 364)
Training targets: (3367,)
Validation targets: (842,)


# Scale the features before PCA

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Learn the scaling using only the training portion.
X_train_scaled = scaler.fit_transform(X_train)

# Apply that same scaling to the validation portion.
X_valid_scaled = scaler.transform(X_valid)

print("Scaled training features:", X_train_scaled.shape)
print("Scaled validation features:", X_valid_scaled.shape)

Scaled training features: (3367, 364)
Scaled validation features: (842, 364)


# We apply PCA to reduce the number of features, retaining at least 95% of the scaled training data’s variance.

In [9]:
from sklearn.decomposition import PCA

# Keep enough components to retain at least 95% of the variance.
pca = PCA(n_components=0.95, svd_solver="full")

# Learn the components from training rows only.
X_train_pca = pca.fit_transform(X_train_scaled)

# Apply the same transformation to validation rows.
X_valid_pca = pca.transform(X_valid_scaled)

print("Features before PCA:", X_train_scaled.shape[1])
print("Components after PCA:", pca.n_components_)
print(
    "Variance retained:",
    f"{pca.explained_variance_ratio_.sum():.2%}"
)
print("Training shape after PCA:", X_train_pca.shape)
print("Validation shape after PCA:", X_valid_pca.shape)

Features before PCA: 364
Components after PCA: 145
Variance retained: 95.01%
Training shape after PCA: (3367, 145)
Validation shape after PCA: (842, 145)


# We train an XGBoost regressor to predict test-bench time in seconds.

In [10]:
from xgboost import XGBRegressor

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

# Learn from the reduced training features and known times.
model.fit(X_train_pca, y_train)

# Predict times for the held-out validation rows.
validation_predictions = model.predict(X_valid_pca)

print("Training complete!")
print("Validation predictions:", len(validation_predictions))
print("First five predictions:", validation_predictions[:5])

Training complete!
Validation predictions: 842
First five predictions: [ 97.171616  95.56224  108.10079   77.41205  108.268585]


In [11]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# A simple baseline predicts the training average for every row.
baseline_predictions = np.full(len(y_valid), y_train.mean())

results = pd.DataFrame({
    "Model": ["Mean baseline", "PCA + XGBoost"],
    "MAE (seconds)": [
        mean_absolute_error(y_valid, baseline_predictions),
        mean_absolute_error(y_valid, validation_predictions)
    ],
    "RMSE (seconds)": [
        np.sqrt(mean_squared_error(y_valid, baseline_predictions)),
        np.sqrt(mean_squared_error(y_valid, validation_predictions))
    ],
    "R-squared": [
        r2_score(y_valid, baseline_predictions),
        r2_score(y_valid, validation_predictions)
    ]
})

display(results.round(4))

,Model,MAE (seconds),RMSE (seconds),R-squared
0,Mean baseline,10.1426,12.4762,-0.0000
1,PCA + XGBoost,5.9595,8.6864,0.5152


In [12]:
# Ensure test columns match the training column order.
X_test_encoded = X_test_encoded[X_encoded.columns]

# Learn scaling from all labeled training rows.
final_scaler = StandardScaler()
X_full_scaled = final_scaler.fit_transform(X_encoded)
X_test_scaled = final_scaler.transform(X_test_encoded)

# Learn PCA from all labeled training rows.
final_pca = PCA(n_components=0.95, svd_solver="full")
X_full_pca = final_pca.fit_transform(X_full_scaled)
X_test_pca = final_pca.transform(X_test_scaled)

print("Final training shape:", X_full_pca.shape)
print("Final test shape:", X_test_pca.shape)
print(
    "Variance retained:",
    f"{final_pca.explained_variance_ratio_.sum():.2%}"
)

Final training shape: (4209, 148)
Final test shape: (4209, 148)
Variance retained: 95.00%


In [13]:
# Reuse the model settings we evaluated.
final_model = XGBRegressor(**model.get_params())

# Train on all labeled rows.
final_model.fit(X_full_pca, y)

# Predict test-bench times in seconds.
test_predictions = final_model.predict(X_test_pca)

print("Number of test predictions:", len(test_predictions))
print("First five predictions:", test_predictions[:5])

Number of test predictions: 4209
First five predictions: [104.04141  122.60365  107.615234  84.707565 104.99014 ]


# Pair each car with its predicted test-bench time.

In [14]:
# Pair each car ID with its predicted test-bench time.
predictions_df = pd.DataFrame({
    "ID": test_ids.to_numpy(),
    "y": test_predictions
})

# Check that every test row has a valid prediction.
assert len(predictions_df) == len(test_df)
assert np.isfinite(predictions_df["y"]).all()

# Save without adding an extra index column.
predictions_df.to_csv("mercedes_predictions.csv", index=False)

print("Saved predictions:", predictions_df.shape)
display(predictions_df.head())

# Download the CSV to your computer.
from google.colab import files
files.download("mercedes_predictions.csv")

Saved predictions: (4209, 2)


,ID,y
0,1,104.041412
1,2,122.603653
2,3,107.615234
3,4,84.707565
4,5,104.990143


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Project Conclusion

I developed an XGBoost regression model to predict Mercedes-Benz
test-bench time in seconds.

### Data preparation
- Checked missing values and unique-value counts in both datasets.
- Confirmed that neither dataset contained missing values.
- Excluded ID from the predictors and separated the target variable y.
- Removed 12 zero-variance training features from both datasets,
  leaving 364 predictors.
- Applied label encoding to eight categorical columns using shared
  category mappings fitted on the combined train and test categories.

### Dimensionality reduction
I standardized the features before applying PCA. For validation,
PCA retained 145 components and approximately 95.01% of the variance.
Scaling and PCA were fitted on the training portion and applied
to the validation portion.

### Model evaluation
Using an 80/20 training-validation split, PCA plus XGBoost achieved:

| Metric | Mean baseline | PCA + XGBoost |
|---|---:|---:|
| MAE (seconds) | 10.1426 | 5.9595 |
| RMSE (seconds) | 12.4762 | 8.6864 |
| R-squared | Approximately 0 | 0.5152 |

The model improved on predicting the average training time.
Its average absolute validation error was approximately 5.96 seconds.

### Final predictions
I refitted scaling and PCA using all 4,209 labeled training rows.
The final PCA retained 148 components and approximately 95% of
feature variance. I trained the final XGBoost model and saved
4,209 predictions in mercedes_predictions.csv.

### Limitations
Test-set performance cannot be measured because test.csv has no
actual target values. Label encoding introduces artificial numeric
ordering, and retaining 95% of feature variance does not imply
95% prediction accuracy. The results do not directly demonstrate
reductions in testing time or emissions.